In [1]:
"""
Day 3 — HRTF Baseline
SpatialMesh: Apply HRTF to a voice and hear it from a specific direction

"""

'\nDay 3 — HRTF Baseline\nSpatialMesh: Apply HRTF to a voice and hear it from a specific direction\n\n'

In [2]:
import numpy as np
import soundfile as sf
import sofar as sf_sofa
from scipy.signal import fftconvolve
import os   

In [17]:
SOFA_PATH = r"E:\Hackathons\Samsung AX Challange\data\sonicom\P0001\HRTF\HRTF\48kHz\P0001_FreeFieldComp_48kHz.sofa"
AUDIO_PATH = r"E:\Hackathons\Samsung AX Challange\data\librispeech\test-clean\LibriSpeech\test-clean\61\70968\61-70968-0000.flac"
OUTPUT_PATH = r"E:\Hackathons\Samsung AX Challange\output_spatial.wav"

In [4]:
def load_hrtf(sofa_path):
    """Load HRTF from SOFA file and return positions + impulse responses"""
    hrtf = sf_sofa.read_sofa(sofa_path)

    positions = hrtf.SourcePosition # gives us the positions of sources i.e there are 793 positions and (radius,azimuthal angle, elevation angle) are arguments

    # Data_IR shape: (num_positions(793), ear(2), num_samples(256) = z) =HRIR Data (Time Domain Samples of HRTF)
    # Axis 1: 0=left ear, 1= right ear
    ir = hrtf.Data_IR
     
    
    sample_rate = int(hrtf.Data_SamplingRate)
    
    print(f"HRTF loaded: {positions.shape[0]} positions")
    print(f"IR length: {ir.shape[2]} samples")
    print(f"Sample rate: {sample_rate} Hz")
    return positions, ir, sample_rate


In [5]:
def find_hrtf_for_direction(positions, ir, target_azimuth, target_elevation): 
    #Basically we're finding the HRTF_Left and right based on given azimuth and elevation angle
    az_rad = np.radians(positions[:, 0])
    el_rad = np.radians(positions[:, 1])
    tgt_az = np.radians(target_azimuth)
    tgt_el = np.radians(target_elevation)
    
    closest_idx = np.argmax(np.clip(
        np.sin(el_rad)*np.sin(tgt_el) +
        np.cos(el_rad)*np.cos(tgt_el)*np.cos(az_rad - tgt_az),
        -1.0, 1.0
    ))
    actual_az = positions[closest_idx, 0]
    actual_el = positions[closest_idx, 1]
    
    print(f"Requested: az={target_azimuth}° el={target_elevation}°")
    print(f"Closest found: az={actual_az}° el={actual_el}°")
    
    hrtf_left  = ir[closest_idx, 1, :]   # SONICOM: channel 1 = left ear
    hrtf_right = ir[closest_idx, 0, :]   # SONICOM: channel 0 = right ear
    
    return hrtf_left, hrtf_right

In [6]:
def apply_hrtf(mono_audio, hrtf_left, hrtf_right):
    """
    Convolve mono audio with HRTF to create binaural (stereo) output.
    This is the core operation that creates the spatial illusion.
    """
    left_channel  = fftconvolve(mono_audio, hrtf_left,  mode='full')
    right_channel = fftconvolve(mono_audio, hrtf_right, mode='full')
    
    # Normalize to prevent clipping
    max_val = max(np.max(np.abs(left_channel)), 
                  np.max(np.abs(right_channel)))
    if max_val > 0:
        left_channel  = left_channel  / max_val * 0.9
        right_channel = right_channel / max_val * 0.9
    
    # Stack into stereo: shape (samples, 2)
    binaural = np.stack([left_channel, right_channel], axis=1)
    return binaural


In [7]:
def render_voice_at_position(audio_path, sofa_path, output_path,
                              azimuth, elevation):
    """
    Full pipeline: load voice + HRTF, render spatial audio, save.
    
    azimuth:   90  = LEFT
               270 = RIGHT  
               0   = FRONT
               180 = BACK
    elevation: 0   = ear level
    """
    print("\n" + "="*50)
    print(f"Rendering voice at azimuth={azimuth}° elevation={elevation}°")
    print("="*50)
    
    # Load voice audio
    audio, audio_sr = sf.read(audio_path)
    
    # Convert stereo to mono if needed
    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)
    
    print(f"Voice loaded: {len(audio)/audio_sr:.2f} seconds at {audio_sr}Hz")
    
    # Load HRTF
    positions, ir, hrtf_sr = load_hrtf(sofa_path)
    
    # Resample audio if sample rates don't match
    if audio_sr != hrtf_sr:
        from scipy.signal import resample
        target_len = int(len(audio) * hrtf_sr / audio_sr)
        audio = resample(audio, target_len)
        print(f"Resampled audio from {audio_sr}Hz to {hrtf_sr}Hz")
    
    # Get HRTF for target direction
    hrtf_L, hrtf_R = find_hrtf_for_direction(
        positions, ir, azimuth, elevation
    )
    # Quick ITD/ILD sanity check
    corr = np.correlate(hrtf_L, hrtf_R, mode='full')
    lag = int(np.argmax(np.abs(corr))) - (len(hrtf_R) - 1)
    itd_us = lag / hrtf_sr * 1_000_000

    rms_L = np.sqrt(np.mean(hrtf_L**2))
    rms_R = np.sqrt(np.mean(hrtf_R**2))
    ild_db = 20 * np.log10((rms_L + 1e-12) / (rms_R + 1e-12))

    print(f"  ITD: {itd_us:+.1f} µs  |  ILD: {ild_db:+.2f} dB")
    # Apply HRTF — this creates the 3D illusion
    binaural = apply_hrtf(audio, hrtf_L, hrtf_R)
    
    # Save output
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    sf.write(output_path, binaural, hrtf_sr)
    print(f"\nSaved to: {output_path}")
    print("Put on headphones and play this file!")
    
    return binaural, hrtf_sr


In [8]:
def test_three_directions():
    """
    Test the core milestone: render same voice from LEFT, FRONT, RIGHT
    Compare all three to verify spatial rendering works.
    """
    directions = [
        ("LEFT",  90,  0),
        ("FRONT",  0,  0),
        ("RIGHT", 270,  0),
    ]
    
    base = r"E:\Hackathons\Samsung AX Challange"
    
    for name, az, el in directions:
        output = os.path.join(base, f"output_{name.lower()}.wav")
        render_voice_at_position(
            audio_path=AUDIO_PATH,
            sofa_path=SOFA_PATH,
            output_path=output,
            azimuth=az,
            elevation=el
        )
        print(f"✓ {name} rendered\n")
    
    print("\n" + "="*50)
    print("ALL THREE DIRECTIONS RENDERED")
    print("="*50)
    print("Play these files with headphones:")
    print(f"  output_left.wav  → voice should sound LEFT")
    print(f"  output_front.wav → voice should sound FRONT")
    print(f"  output_right.wav → voice should sound RIGHT")
    print("\nIf you can tell the difference → Day 3 SUCCESS ✅")


In [9]:
if __name__ == "__main__":
    test_three_directions()


Rendering voice at azimuth=90° elevation=0°
Voice loaded: 4.91 seconds at 16000Hz
SOFA file contained custom entries
----------------------------------
GLOBAL_ReceiverDescription, GLOBAL_RoomDescription, GLOBAL_RoomLocation, GLOBAL_SourceDescription, GLOBAL_EmitterDescription, MeasurementSourceAudioChannel
HRTF loaded: 793 positions
IR length: 2400 samples
Sample rate: 48000 Hz
Resampled audio from 16000Hz to 48000Hz
Requested: az=90° el=0°
Closest found: az=90.0° el=0.0°
  ITD: +791.7 µs  |  ILD: -18.03 dB

Saved to: E:\Hackathons\Samsung AX Challange\output_left.wav
Put on headphones and play this file!
✓ LEFT rendered


Rendering voice at azimuth=0° elevation=0°
Voice loaded: 4.91 seconds at 16000Hz
SOFA file contained custom entries
----------------------------------
GLOBAL_ReceiverDescription, GLOBAL_RoomDescription, GLOBAL_RoomLocation, GLOBAL_SourceDescription, GLOBAL_EmitterDescription, MeasurementSourceAudioChannel
HRTF loaded: 793 positions
IR length: 2400 samples
Sample rat

In [10]:
import os

path = r"E:\Hackathons\Samsung AX Challange\data\librispeech\test-clean\LibriSpeech\test-clean\61\70968\61-70968-0000.flac"

print(f"File exists: {os.path.exists(path)}")
print(f"File size: {os.path.getsize(path) if os.path.exists(path) else 'N/A'} bytes")

File exists: True
File size: 97158 bytes


In [11]:
import numpy as np
import sofar as sf_sofa
import matplotlib.pyplot as plt

hrtf = sf_sofa.read_sofa(SOFA_PATH)
positions = hrtf.SourcePosition
ir = hrtf.Data_IR
fs = int(hrtf.Data_SamplingRate)

# Find 90° (LEFT) and 270° (RIGHT)
for target_az in [0, 90, 180, 270]:
    az_diff = np.abs(positions[:, 0] - target_az)
    idx = np.argmin(az_diff)
    L = ir[idx, 1, :]
    R = ir[idx, 0, :]
    
    corr = np.correlate(L, R, mode='full')
    lag = int(np.argmax(np.abs(corr))) - (len(R) - 1)
    itd_us = lag / fs * 1_000_000
    rms_L = np.sqrt(np.mean(L**2))
    rms_R = np.sqrt(np.mean(R**2))
    ild_db = 20 * np.log10((rms_L + 1e-12) / (rms_R + 1e-12))
    
    print(f"Az={target_az:3.0f}°  →  ITD={itd_us:+6.1f}µs   ILD={ild_db:+5.2f}dB")

SOFA file contained custom entries
----------------------------------
GLOBAL_ReceiverDescription, GLOBAL_RoomDescription, GLOBAL_RoomLocation, GLOBAL_SourceDescription, GLOBAL_EmitterDescription, MeasurementSourceAudioChannel
Az=  0°  →  ITD= +41.7µs   ILD=-2.46dB
Az= 90°  →  ITD=+104.2µs   ILD=-7.31dB
Az=180°  →  ITD= -41.7µs   ILD=-0.06dB
Az=270°  →  ITD=-437.5µs   ILD=+17.98dB


In [12]:
# Verify the fix selects the correct index
az_rad = np.radians(positions[:, 0])
el_rad = np.radians(positions[:, 1])

for target_az, target_el in [(0,0), (90,0), (180,0), (270,0)]:
    tgt_az = np.radians(target_az)
    tgt_el = np.radians(target_el)
    
    distances = np.arccos(np.clip(
        np.sin(el_rad)*np.sin(tgt_el) +
        np.cos(el_rad)*np.cos(tgt_el)*np.cos(az_rad - tgt_az),
        -1.0, 1.0
    ))
    
    idx = np.argmin(distances)
    L = ir[idx, 1, :]
    R = ir[idx, 0, :]
    
    corr = np.correlate(L, R, mode='full')
    lag = int(np.argmax(np.abs(corr))) - (len(R) - 1)
    itd_us = lag / fs * 1_000_000
    rms_L = np.sqrt(np.mean(L**2))
    rms_R = np.sqrt(np.mean(R**2))
    ild_db = 20 * np.log10((rms_L + 1e-12) / (rms_R + 1e-12))
    
    print(f"Az={target_az:3d}°  idx={idx}  El={positions[idx,1]:.0f}°  →  ITD={itd_us:+6.1f}µs   ILD={ild_db:+5.2f}dB")

Az=  0°  idx=4  El=0°  →  ITD= +62.5µs   ILD=-5.08dB
Az= 90°  idx=414  El=0°  →  ITD=+791.7µs   ILD=-18.03dB
Az=180°  idx=18  El=0°  →  ITD= -83.3µs   ILD=-0.11dB
Az=270°  idx=401  El=0°  →  ITD=-666.7µs   ILD=+17.10dB


In [13]:
import soundfile as sf
import numpy as np

left_audio, _ = sf.read(r"E:\Hackathons\Samsung AX Challange\output_left.wav")
right_audio, _ = sf.read(r"E:\Hackathons\Samsung AX Challange\output_right.wav")

print("LEFT file:")
print(f"  Ch0 RMS: {np.sqrt(np.mean(left_audio[:,0]**2)):.6f}")
print(f"  Ch1 RMS: {np.sqrt(np.mean(left_audio[:,1]**2)):.6f}")
print(f"  Ch0==Ch1: {np.allclose(left_audio[:,0], left_audio[:,1])}")

print("\nRIGHT file:")
print(f"  Ch0 RMS: {np.sqrt(np.mean(right_audio[:,0]**2)):.6f}")
print(f"  Ch1 RMS: {np.sqrt(np.mean(right_audio[:,1]**2)):.6f}")
print(f"  Ch0==Ch1: {np.allclose(right_audio[:,0], right_audio[:,1])}")

print("\nLEFT vs RIGHT files identical?", np.allclose(left_audio, right_audio))

LEFT file:
  Ch0 RMS: 0.028342
  Ch1 RMS: 0.051432
  Ch0==Ch1: False

RIGHT file:
  Ch0 RMS: 0.053728
  Ch1 RMS: 0.029040
  Ch0==Ch1: False

LEFT vs RIGHT files identical? False


In [14]:
import numpy as np
import soundfile as sf

# Load the original mono audio
audio, fs = sf.read(AUDIO_PATH)
if audio.ndim == 2:
    audio = np.mean(audio, axis=1)
audio = audio / np.abs(audio).max()

# Create an OBVIOUSLY different stereo file
# Left channel = normal audio, Right channel = silence
obvious_left = np.stack([audio, np.zeros_like(audio)], axis=1)
sf.write(r"E:\Hackathons\Samsung AX Challange\test_obvious_left.wav", obvious_left, fs)

# Left channel = silence, Right channel = normal audio  
obvious_right = np.stack([np.zeros_like(audio), audio], axis=1)
sf.write(r"E:\Hackathons\Samsung AX Challange\test_obvious_right.wav", obvious_right, fs)

print("Written. Play both files with headphones.")
print("test_obvious_left.wav  → you should hear ONLY in left ear")
print("test_obvious_right.wav → you should hear ONLY in right ear")

Written. Play both files with headphones.
test_obvious_left.wav  → you should hear ONLY in left ear
test_obvious_right.wav → you should hear ONLY in right ear


In [15]:
import soundfile as sf
import numpy as np

# Verify the file on disk is actually stereo
audio, fs = sf.read(r"E:\Hackathons\Samsung AX Challange\test_obvious_left.wav")
print(f"Shape: {audio.shape}")
print(f"Left  channel RMS: {np.sqrt(np.mean(audio[:,0]**2)):.6f}")
print(f"Right channel RMS: {np.sqrt(np.mean(audio[:,1]**2)):.6f}")
print(f"Right is silence: {np.allclose(audio[:,1], 0)}")

Shape: (78480, 2)
Left  channel RMS: 0.131039
Right channel RMS: 0.000000
Right is silence: True


In [16]:
import numpy as np
import soundfile as sf
from scipy.signal import fftconvolve, resample

def render_pan_sweep():
    """
    Render same voice from 9 positions: 270° → 180° → 90°
    (RIGHT → BACK → LEFT in SONICOM convention)
    Concatenate into one file so you hear the pan sweep.
    """
    audio, audio_sr = sf.read(AUDIO_PATH)
    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)
    audio = audio / np.abs(audio).max()

    positions_data = hrtf.SourcePosition
    ir_data = hrtf.Data_IR
    fs = int(hrtf.Data_SamplingRate)

    # Resample audio to HRTF sample rate
    if audio_sr != fs:
        audio = resample(audio, int(len(audio) * fs / audio_sr))

    # Trim to 1.5 seconds per position so sweep isn't too long
    trim = int(1.5 * fs)
    audio = audio[:trim]

    az_rad = np.radians(positions_data[:, 0])
    el_rad = np.radians(positions_data[:, 1])

    # 9 positions: right → front → left
    azimuths = [270, 247, 225, 180, 135, 112, 90]
    
    segments = []
    for az in azimuths:
        tgt_az = np.radians(az)
        tgt_el = np.radians(0)
        
        distances = np.arccos(np.clip(
            np.sin(el_rad)*np.sin(tgt_el) +
            np.cos(el_rad)*np.cos(tgt_el)*np.cos(az_rad - tgt_az),
            -1.0, 1.0
        ))
        
        idx = np.argmin(distances)
        L = ir_data[idx, 1, :]
        R = ir_data[idx, 0, :]
        
        left_ch  = fftconvolve(audio, L, mode='full')
        right_ch = fftconvolve(audio, R, mode='full')
        
        peak = max(np.abs(left_ch).max(), np.abs(right_ch).max())
        left_ch  = left_ch  / peak * 0.9
        right_ch = right_ch / peak * 0.9
        
        stereo = np.stack([left_ch, right_ch], axis=1)
        segments.append(stereo)
        print(f"Az={az:3d}°  idx={idx}  El={positions_data[idx,1]:.0f}°  rendered")
    
    # Add 0.3s silence between positions so you can clearly hear each one
    silence = np.zeros((int(0.3 * fs), 2))
    sweep = []
    for seg in segments:
        sweep.append(seg)
        sweep.append(silence)
    
    sweep_audio = np.concatenate(sweep, axis=0)
    
    out_path = r"E:\Hackathons\Samsung AX Challange\output_sweep.wav"
    sf.write(out_path, sweep_audio, fs)
    print(f"\nSaved → {out_path}")
    print("Play with headphones — voice should move RIGHT → FRONT → LEFT")

# Make sure hrtf is loaded first
hrtf = sf_sofa.read_sofa(SOFA_PATH)
render_pan_sweep()

SOFA file contained custom entries
----------------------------------
GLOBAL_ReceiverDescription, GLOBAL_RoomDescription, GLOBAL_RoomLocation, GLOBAL_SourceDescription, GLOBAL_EmitterDescription, MeasurementSourceAudioChannel
Az=270°  idx=401  El=0°  rendered
Az=247°  idx=511  El=0°  rendered
Az=225°  idx=599  El=0°  rendered
Az=180°  idx=18  El=0°  rendered
Az=135°  idx=216  El=0°  rendered
Az=112°  idx=326  El=0°  rendered
Az= 90°  idx=414  El=0°  rendered

Saved → E:\Hackathons\Samsung AX Challange\output_sweep.wav
Play with headphones — voice should move RIGHT → FRONT → LEFT
